In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Paths
data_path = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray"

train_path = os.path.join(data_path, "train")
val_path = os.path.join(data_path, "val")
test_path = os.path.join(data_path, "test")

# Transforms
train_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

val_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

# Load datasets
train_data = datasets.ImageFolder(train_path, transform=train_transforms)
val_data = datasets.ImageFolder(val_path, transform=val_transforms)
test_data = datasets.ImageFolder(test_path, transform=val_transforms)

# DataLoaders
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

In [3]:
import os
os.makedirs("/kaggle/working/results/transfer", exist_ok=True)

In [4]:
import torch
import torch.nn as nn
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(pretrained=True)

# Freeze layers
for param in model.parameters():
    param.requires_grad = False

# Modify final layer
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

model = model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 169MB/s]


In [5]:
# =========================
# 1. IMPORTS
# =========================
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# 2. PATHS
# =========================
data_path = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray"

train_path = os.path.join(data_path, "train")
val_path = os.path.join(data_path, "val")
test_path = os.path.join(data_path, "test")

# =========================
# 3. RESULTS FOLDER
# =========================
os.makedirs("/kaggle/working/results/experiments", exist_ok=True)

# =========================
# 4. FUNCTION TO RUN EXPERIMENT
# =========================
def run_experiment(name, use_aug=True, use_weights=True, fine_tune=True):

    print(f"\nRunning {name}...\n")

    # Transforms
    if use_aug:
        train_transforms = transforms.Compose([
            transforms.Resize((224,224)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225])
        ])
    else:
        train_transforms = transforms.Compose([
            transforms.Resize((224,224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225])
        ])

    val_transforms = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])

    # Data
    train_data = datasets.ImageFolder(train_path, transform=train_transforms)
    val_data = datasets.ImageFolder(val_path, transform=val_transforms)
    test_data = datasets.ImageFolder(test_path, transform=val_transforms)

    train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

    # Class weights
    if use_weights:
        labels = train_data.targets
        class_counts = Counter(labels)
        total = sum(class_counts.values())
        weights = [total/class_counts[i] for i in range(len(class_counts))]
        weights = torch.tensor(weights).to(device)
        criterion = nn.CrossEntropyLoss(weight=weights)
    else:
        criterion = nn.CrossEntropyLoss()

    # Model
    model = models.resnet18(pretrained=True)

    for param in model.parameters():
        param.requires_grad = False

    # Fine-tuning control
    if fine_tune:
        for param in model.layer4.parameters():
            param.requires_grad = True

    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, 2)

    model = model.to(device)

    optimizer = optim.AdamW(model.parameters(), lr=0.0003)

    # Training
    epochs = 5

    for epoch in range(epochs):
        model.train()
        running_loss = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()

        print(f"{name} Epoch {epoch+1} Loss: {running_loss/len(train_loader):.4f}")

    # Evaluation
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds)

    print(f"{name} Accuracy: {acc}")

    # Save result
    with open(f"/kaggle/working/results/experiments/{name}.txt", "w") as f:
        f.write(f"{name} Accuracy: {acc}")

    return acc


# =========================
# 5. RUN ALL EXPERIMENTS
# =========================
results = {}

results["no_augmentation"] = run_experiment(
    "no_augmentation", use_aug=False, use_weights=True, fine_tune=True
)

results["no_class_weights"] = run_experiment(
    "no_class_weights", use_aug=True, use_weights=False, fine_tune=True
)

# 🔥 UPDATED: WITH FINE-TUNING (instead of no_finetuning)
results["with_finetuning"] = run_experiment(
    "with_finetuning", use_aug=True, use_weights=True, fine_tune=True
)

print("\nFINAL RESULTS:")
print(results)


Running no_augmentation...



/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


no_augmentation Epoch 1 Loss: 0.0998
no_augmentation Epoch 2 Loss: 0.0232
no_augmentation Epoch 3 Loss: 0.0139
no_augmentation Epoch 4 Loss: 0.0176
no_augmentation Epoch 5 Loss: 0.0093
no_augmentation Accuracy: 0.8141025641025641

Running no_class_weights...



/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


no_class_weights Epoch 1 Loss: 0.1229
no_class_weights Epoch 2 Loss: 0.0688
no_class_weights Epoch 3 Loss: 0.0631
no_class_weights Epoch 4 Loss: 0.0533
no_class_weights Epoch 5 Loss: 0.0374
no_class_weights Accuracy: 0.8830128205128205

Running with_finetuning...



/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


with_finetuning Epoch 1 Loss: 0.1503
with_finetuning Epoch 2 Loss: 0.0820
with_finetuning Epoch 3 Loss: 0.0686
with_finetuning Epoch 4 Loss: 0.0653
with_finetuning Epoch 5 Loss: 0.0510
with_finetuning Accuracy: 0.8733974358974359

FINAL RESULTS:
{'no_augmentation': 0.8141025641025641, 'no_class_weights': 0.8830128205128205, 'with_finetuning': 0.8733974358974359}
